In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

### 导入必要的map参数

In [2]:
# 物流的渠道对照关系清洗用
month = 202512
Channel_map = {
'工程': '工程',
'零售':'零售',
'电商不可售':'电商',
'电商':'电商',
'内部处理通用':'电商',
'借出渠道':'非零售工程电商',
'新品':'电商',
'战略电商':'电商',
'转出渠道':'非零售工程电商',
'每誉':'每誉',
'渠道':'非零售工程电商',
'海外':'海外',
'调出渠道':'非零售工程电商',
'非零售工程电商':'非零售工程电商',
'非零售工程电商':'非零售工程电商'
}
#用于合并计算
productgroupset_map = {
    '吸油烟机':['吸油烟机'],
    '灶具':['灶具'],
    '烤箱':['烤箱'],
    '蒸箱':['蒸箱'],
    '微波炉':['微波炉'],
    '蒸烤烹饪机':['蒸烤烹饪机'],
    '蒸烤微烹饪机':['蒸烤微烹饪机'],
    '蒸微':['蒸微'],
    '蒸烤微合计':['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶消烹饪机':['灶消烹饪机'],
    '灶蒸烹饪机':['灶蒸烹饪机'],
    '灶蒸烤烹饪机':['灶蒸烤烹饪机'],
    '灶烤烹饪机':['灶烤烹饪机'],
    '灶集成':['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪产品线合计':['灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜':['消毒柜'],
    '热水器':['热水器'],
    '两用炉':['两用炉'],
    '热水器两用炉合计':['热水器','两用炉'],
    '家用净水机':['家用净水机'],
    '商用净水机':['商用净水机'],
    '净热产品线合计':['热水器','两用炉','家用净水机','商用净水机'],
    '水槽洗碗机':['水槽洗碗机'],
    '嵌入式洗碗机':['嵌入式洗碗机'],
    '洗碗机产品线合计':['水槽洗碗机','嵌入式洗碗机'],
    '国内合计':['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','家用净水机','商用净水机','水槽洗碗机','嵌入式洗碗机'],
}
# 统计的产品组
productgroup_list = ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','热水器两用炉合计','家用净水机','商用净水机','净热产品线合计','水槽洗碗机','嵌入式洗碗机','洗碗机产品线合计','国内合计']


### 数据源处理-合并发货和财务的数据

In [ ]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\物流发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

In [ ]:
#如果有报错请提示报错信息
df = pd.DataFrame()
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        df_temp['发货月份'] = file[0][:8]
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(f'读取{file[0]}的{file[1]}表失败')
    if '实际总数量' in df_temp.columns:
        df_temp = df_temp.rename(columns={'实际总数量':'实际出库数量'})
    df_temp = df_temp[['商品编码', '渠道', '实际出库数量','发货月份']]
    df = pd.concat([df, df_temp], axis=0).reset_index(drop=True)
df = df.dropna(how='all').reset_index(drop=True)  # 仅当一行所有值都是NaN时才删除
df['商品编码'] = df['商品编码'].astype(str)
df['渠道'].value_counts()

In [ ]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\财务发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

In [ ]:
caiwu_shouru = {'商品编码':[],'渠道':[],'实际出库数量':[],'发货月份':[]}
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(file[0], file[1])
    for index,row in df_temp.iterrows():
        if row['零售'] > 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('零售')
            caiwu_shouru['实际出库数量'].append(row['零售'])
            caiwu_shouru['发货月份'].append(file[0][:8])
        if row['工程'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('工程')
            caiwu_shouru['实际出库数量'].append(row['工程'])
            caiwu_shouru['发货月份'].append(file[0][:8])
        if row['电商'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('电商')
            caiwu_shouru['实际出库数量'].append(row['电商'])
            caiwu_shouru['发货月份'].append(file[0][:8])
        if row['合计-发货'] - row['零售'] - row['工程'] - row['电商'] != 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('非零售工程电商')
            caiwu_shouru['实际出库数量'].append(row['合计-发货'] - row['零售'] - row['工程'] - row['电商'])
            caiwu_shouru['发货月份'].append(file[0][:8])
df_caiwu = pd.DataFrame(caiwu_shouru)
df_caiwu['商品编码'] = df_caiwu['商品编码'].astype(str)
df_caiwu['渠道'].value_counts()


In [ ]:
df0 = pd.concat([df, df_caiwu], axis=0).reset_index(drop=True)
# df0.to_excel(r'C:\Users\zhangbon\Desktop\总发货.xlsx', index=False)
df0['物料编码'] = df0['商品编码'].astype(str)
df0['渠道'] = df0['渠道'].map(Channel_map).fillna('非零售工程电商')
df0 = df0[(df0['渠道']!='无') & 
        (df0['物料编码'].str.len()>=10)
        ].reset_index(drop=True)
df0['渠道'].value_counts()
# df0

### 数据处理-添加核算价，产品线、产品组、标准型号、销售时间、销售渠道、物料组（大系列）等信息

In [ ]:
df0  = df0[['物料编码','渠道','实际出库数量','发货月份']]
df0['发货月份'] = df0['发货月份'].str.replace('明','')
df0['发货月份'] = pd.to_datetime(df0['发货月份'],format=r'%Y年%m月')
df0


In [6]:
# 这里用于替换数据源
df0 = pd.read_excel(fr'C:\Users\zhangbon\Desktop\总仓发货数据明细（财务口径）_图表1 (7).xlsx')
df0['物料编码'] = df0['物料编码'].astype(str).str[:13]
df0['渠道'] = df0['业务线']
df0['实际出库数量'] = df0['总仓发货数量']
df0['发货月份'] = pd.to_datetime(df0['年月']).dt.strftime('%Y-%m')
df0 = df0[['物料编码','渠道','实际出库数量','发货月份']]
df0 = df0[df0['实际出库数量'] > 0]
df0

,物料编码,渠道,实际出库数量,发货月份
0,1101000300440,零售,2,2019-01
2,1101000502070,零售,273,2019-01
4,110100050209A,零售,165,2019-01
6,1101000516730,零售,1484,2019-01
8,110100051870A,零售,936,2019-01
...,...,...,...,...
117468,1001001500116,零售,2814,2025-11
117471,1001001500130,零售,3415,2025-11
117473,1001001500130,电商,6,2025-11
117474,1001001500098,零售,684,2025-11


In [7]:
# 引入物料组（大系列）
df_bigtype = pd.read_excel(r'C:\Users\zhangbon\Desktop\物料组对照（大系列）.xlsx')
df_bigtype['物料组'] = df_bigtype['物料组'].astype(str)
bigtype_map = dict(zip(df_bigtype['物料组'], df_bigtype['物料组描述']))
# bigtype_map

In [8]:
# PLM产品数据导入
df_plm = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_plm[['物料号','标准型号','国内/海外']] = df_plm[['物料号','标准型号','国内/海外']].astype(str)
df_plm['物料编码'] = df_plm['物料号'].apply(lambda x: x[:13])
df_plm['渠道'] = df_plm['下属渠道']
df_plm = df_plm[['物料编码','产品型号','标准型号','国内/海外','产品组','产品线','产品状态','渠道','开始销售时间','对应渠道状态','停止销售时间']]
df_plm


e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,物料编码,产品型号,标准型号,国内/海外,产品组,产品线,产品状态,渠道,开始销售时间,对应渠道状态,停止销售时间
0,nan,02-CS34BW,02-CS34BW,国内,灶具,烹饪厨电产品线,开发,零售,NaN,未售,NaN
1,nan,02-CS34BW,02-CS34BW,国内,灶具,烹饪厨电产品线,开发,电商,NaN,未售,NaN
2,nan,02-CS34BW,02-CS34BW,国内,灶具,烹饪厨电产品线,开发,工程,NaN,未售,NaN
3,nan,1,nan,国内,水槽洗碗机,洗碗机产品线,作废,NaN,NaN,NaN,NaN
4,1004000200090,10T-JSG15-0606FR,JSG15-0606,国内,热水器,净热产品线,停止发货,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
9306,nan,苍穹V6R1,苍穹V6R1,国内,吸油烟机,油烟机产品线,开发,NaN,NaN,NaN,NaN
9307,nan,轩辕V1R1,轩辕V1R1,国内,吸油烟机,油烟机产品线,量产,NaN,NaN,NaN,NaN
9308,nan,轩辕V1R1C02,轩辕V1R1C02,国内,吸油烟机,油烟机产品线,样机,NaN,NaN,NaN,NaN
9309,nan,轩辕V1R1C03,轩辕V1R1C03,国内,吸油烟机,油烟机产品线,开发,NaN,NaN,NaN,NaN


In [9]:
# 导入财务的核算价
df_price = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\核算价.xlsx")
df_price['物料编码'] = df_price['产品编码'].astype(str).str[:13]
df_price['系统核算价'] = df_price['系统核算价'].astype(float)
# df_price
df_price = df_price[['物料编码','系统核算价']]
df_price 

,物料编码,系统核算价
0,1001000100033,2056.0
1,1001000100034,1253.0
2,1001000100035,704.0
3,1001000100036,1990.0
4,1001000100037,1750.0
...,...,...
6736,9102000400001,5000.0
6737,9102000400002,5000.0
6738,1001000900280,2000.0
6739,1008000400014,3500.0


In [10]:
df0 = pd.merge(df0,df_plm,on=['物料编码','渠道'],how='left')
df0 = pd.merge(df0,df_price,on=['物料编码'],how='left')
df0['物料组'] = df0['物料编码'].str[:8]
df0['物料组描述'] = df0['物料组'].map(bigtype_map)
df0['开始销售时间'] = pd.to_datetime(df0['开始销售时间'], format='mixed', errors='coerce')
df0['停止销售时间'] = pd.to_datetime(df0['停止销售时间'], format='mixed', errors='coerce')
# df0[]
df0


,物料编码,渠道,实际出库数量,发货月份,产品型号,标准型号,国内/海外,产品组,产品线,产品状态,开始销售时间,对应渠道状态,停止销售时间,系统核算价,物料组,物料组描述
0,1101000300440,零售,2,2019-01,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,260.0,11010003,零部件油烟机JX系列
1,1101000502070,零售,273,2019-01,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,350.0,11010005,零部件油烟机JQ系列
2,110100050209A,零售,165,2019-01,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,280.0,11010005,零部件油烟机JQ系列
3,1101000516730,零售,1484,2019-01,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,260.0,11010005,零部件油烟机JQ系列
4,110100051870A,零售,936,2019-01,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,260.0,11010005,零部件油烟机JQ系列
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67747,1001001500116,零售,2814,2025-11,CXW-358-Z8T(不带罩),Z8T,国内,吸油烟机,油烟机产品线,停止销售,2022-12-10 12:00:00,停止销售,2025-11-24 16:51:46,3358.0,10010015,整机油烟机Z系列
67748,1001001500130,零售,3415,2025-11,CXW-358-Z9T（不带罩）,Z9T,国内,吸油烟机,油烟机产品线,量产,2024-05-20 12:00:00,在售,NaT,2428.0,10010015,整机油烟机Z系列
67749,1001001500130,电商,6,2025-11,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,2428.0,10010015,整机油烟机Z系列
67750,1001001500098,零售,684,2025-11,CXW-358-ZD12（不带罩）,ZD11,国内,吸油烟机,油烟机产品线,量产,2023-11-02 12:00:00,在售,NaT,2488.0,10010015,整机油烟机Z系列


### 数据源处理-筛选渠道、产品线、产品组信息

In [11]:
df1 = df0[(df0['国内/海外']=='国内') & 
                # (df0['产品状态'].isin(['开发','停止生产','量产','停止销售','小批量','样机','退市预警','创建',])) &
                (df0['渠道'].isin(['零售','工程','电商']))&
                (df0['产品组'].isin(productgroup_list))
                ].reset_index(drop=True)
df1
df1['核算价'] = df1['系统核算价']*df1['实际出库数量']
print(f'没有系统核算价的是这些数据\n{df1[df1['系统核算价'].isnull()]['物料编码'].drop_duplicates()}')
print(len(df1))
# display(df1)
print('这里把数据单位转为万元')
df1['核算价_万元'] = (pd.to_numeric(df1['核算价'])/10000).round(3)
df1['系统核算价_万元'] = (pd.to_numeric(df1['系统核算价'])/10000).round(3)


没有系统核算价的是这些数据
Series([], Name: 物料编码, dtype: object)
55631
这里把数据单位转为万元


In [12]:
df1.head()
df1.columns

Index(['物料编码', '渠道', '实际出库数量', '发货月份', '产品型号', '标准型号', '国内/海外', '产品组', '产品线',
       '产品状态', '开始销售时间', '对应渠道状态', '停止销售时间', '系统核算价', '物料组', '物料组描述', '核算价',
       '核算价_万元', '系统核算价_万元'],
      dtype='object')

In [14]:
df1.columns
df2 = df1.groupby(['物料编码','发货月份','渠道'],as_index=False).agg(
                                            实际出库数量 = ('实际出库数量','sum'),
                                            产品型号 = ('产品型号','first'),
                                            标准型号 = ('标准型号','first'),
                                            国内海外 = ('国内/海外','first'),
                                            产品组 = ('产品组','first'),
                                            产品线 = ('产品线','first'),
                                            产品状态 = ('产品状态','first'),
                                            对应渠道状态 = ('对应渠道状态','first'),
                                            开始销售时间 = ('开始销售时间','first'),
                                            停止销售时间 = ('停止销售时间','first'),
                                            系统核算价 = ('系统核算价','first'),
                                            物料组 = ('物料组','first'),
                                            物料组描述 = ('物料组描述','first'),
                                            核算价_万元 = ('核算价_万元','sum'),
                                            系统核算价_万元 = ('系统核算价_万元','first'),
                                            )
df2
df2.to_excel('c:\\Users\\zhangbon\\Desktop\\物料-发货月份维度-长.xlsx',index=False)


In [ ]:
df1.columns
df3 = df1.groupby('物料编码',as_index=False).agg(
                                            实际出库数量 = ('实际出库数量','sum'),
                                            产品型号 = ('产品型号','first'),
                                            标准型号 = ('标准型号','first'),
                                            国内海外 = ('国内/海外','first'),
                                            产品组 = ('产品组','first'),
                                            产品线 = ('产品线','first'),
                                            产品状态 = ('产品状态','first'),
                                            开始销售时间 = ('开始销售时间','min'),
                                            停止销售时间 = ('停止销售时间','max'),
                                            系统核算价 = ('系统核算价','first'),
                                            物料组 = ('物料组','first'),
                                            物料组描述 = ('物料组描述','first'),
                                            核算价_万元 = ('核算价_万元','sum'),
                                            系统核算价_万元 = ('系统核算价_万元','first'),
                                            )
df3
df3.to_excel('c:\\Users\\zhangbon\\Desktop\\物料维度.xlsx',index=False)


In [ ]:
df_temp = df2[df2['产品线'] == '油烟机产品线']
df_temp
temp = df_temp.groupby(['物料编码'],as_index=False)['核算价_万元'].sum().reset_index()
# temp['物料编码'].value_counts()
temp['核算价_万元'].quantile([0.25,0.5,0.75,1])
# temp['核算价_万元'].median()
# temp.sort_values(by='核算价_万元',ascending=False).reset_index(drop=True).iloc[563]['核算价_万元']

### 数据分析-产品组维度每个渠道的标准型号数、型号数、核算价总和、单型号贡献

In [ ]:
df_temp1 = df1.copy()
# 创建辅助列
df_temp1['零售渠道标准型号'] = df_temp1.apply(lambda x: x['标准型号'] if x['渠道'] == '零售' else None, axis=1)
df_temp1['工程渠道标准型号'] = df_temp1.apply(lambda x: x['标准型号'] if x['渠道'] == '工程' else None, axis=1)
df_temp1['电商渠道标准型号'] = df_temp1.apply(lambda x: x['标准型号'] if x['渠道'] == '电商' else None, axis=1)

df_temp1['零售渠道核算价_万元'] = df_temp1.apply(lambda x: x['核算价_万元'] if x['渠道'] == '零售' else None, axis=1)
df_temp1['工程渠道核算价_万元'] = df_temp1.apply(lambda x: x['核算价_万元'] if x['渠道'] == '工程' else None, axis=1)
df_temp1['电商渠道核算价_万元'] = df_temp1.apply(lambda x: x['核算价_万元'] if x['渠道'] == '电商' else None, axis=1)

# 然后进行聚合
df_result1 = df_temp1.groupby('产品组', as_index=False).agg(
    标准型号数=('标准型号', 'nunique'),
    型号数=('物料编码', 'nunique'),
    核算价金额总计_万元=('核算价_万元', 'sum'),
    零售渠道标准型号数=('零售渠道标准型号', 'nunique'),
    零售渠道核算价_万元=('零售渠道核算价_万元', 'sum'),
    工程渠道标准型号数=('工程渠道标准型号', 'nunique'),
    工程渠道核算价_万元=('工程渠道核算价_万元', 'sum'),
    电商渠道标准型号数=('电商渠道标准型号', 'nunique'),
    电商渠道核算价_万元=('电商渠道核算价_万元', 'sum')
)
df_result1

In [ ]:
df_result1['产品组'] = pd.Categorical(
    df_result1['产品组'], 
    categories=productgroup_list, 
    ordered=True
)
df_result1 = df_result1.sort_values('产品组').reset_index(drop=True)
df_result1

### 数据分析-只看吸油烟机，看各个系列的单型号情况

In [ ]:
df_temp2 = df1.copy()
df_temp2 = df_temp2[df_temp2['产品线']=='油烟机产品线']


#### 从物料组（大系列）角度分析

In [ ]:
# 创建辅助列
df_temp2['零售渠道标准型号'] = df_temp2.apply(lambda x: x['标准型号'] if x['渠道'] == '零售' else None, axis=1)
df_temp2['工程渠道标准型号'] = df_temp2.apply(lambda x: x['标准型号'] if x['渠道'] == '工程' else None, axis=1)
df_temp2['电商渠道标准型号'] = df_temp2.apply(lambda x: x['标准型号'] if x['渠道'] == '电商' else None, axis=1)

df_temp2['零售渠道核算价_万元'] = df_temp2.apply(lambda x: x['核算价_万元'] if x['渠道'] == '零售' else None, axis=1)
df_temp2['工程渠道核算价_万元'] = df_temp2.apply(lambda x: x['核算价_万元'] if x['渠道'] == '工程' else None, axis=1)
df_temp2['电商渠道核算价_万元'] = df_temp2.apply(lambda x: x['核算价_万元'] if x['渠道'] == '电商' else None, axis=1)

df_temp2['零售渠道发货量'] = df_temp2.apply(lambda x: x['实际出库数量'] if x['渠道'] == '零售' else None, axis=1)
df_temp2['工程渠道发货量'] = df_temp2.apply(lambda x: x['实际出库数量'] if x['渠道'] == '工程' else None, axis=1)
df_temp2['电商渠道发货量'] = df_temp2.apply(lambda x: x['实际出库数量'] if x['渠道'] == '电商' else None, axis=1)
# 然后进行聚合
df_result2 = df_temp2.groupby('物料组描述', as_index=False).agg(
    标准型号数=('标准型号', 'nunique'),
    型号数=('物料编码', 'nunique'),
    销售数量=('实际出库数量', 'sum'),
    核算价金额总计_万元=('核算价_万元', 'sum'),
    零售渠道标准型号数=('零售渠道标准型号', 'nunique'),
    零售渠道发货量=('零售渠道发货量', 'sum'),
    零售渠道核算价_万元=('零售渠道核算价_万元', 'sum'),
    工程渠道标准型号数=('工程渠道标准型号', 'nunique'),
    工程渠道发货量=('工程渠道发货量', 'sum'),
    工程渠道核算价_万元=('工程渠道核算价_万元', 'sum'),
    电商渠道标准型号数=('电商渠道标准型号', 'nunique'),
    电商渠道发货量=('电商渠道发货量', 'sum'),
    电商渠道核算价_万元=('电商渠道核算价_万元', 'sum')
)
df_result2['单型号贡献'] = df_result2['核算价金额总计_万元'] / df_result2['标准型号数']
df_result2['零售渠道单型号贡献平均值'] = (df_result2['零售渠道核算价_万元'] / df_result2['零售渠道标准型号数']).fillna(0)
df_result2['工程渠道单型号贡献平均值'] = (df_result2['工程渠道核算价_万元'] / df_result2['工程渠道标准型号数']).fillna(0)
df_result2['电商渠道单型号贡献平均值'] = (df_result2['电商渠道核算价_万元'] / df_result2['电商渠道标准型号数']).fillna(0)
df_result2 = df_result2.sort_values(by='核算价金额总计_万元', ascending=False).reset_index(drop=True)
df_result2

#### 从2080的角度分析

In [ ]:
# 从2080的角度分析一下
df_result3 = df_temp2.groupby(['物料编码'],as_index=False).agg(
                                            产品型号=('产品型号', 'min'),
                                            发货量=('实际出库数量', 'sum'),
                                            核算价金额总计_万元=('核算价_万元', 'sum'),
                                            标准型号=('标准型号', 'min'),
                                            产品状态=('产品状态', 'min'),
                                            物料组=('物料组', 'min'),
                                            物料组描述=('物料组描述', 'min'),
)
df_result3 = df_result3.sort_values(by='核算价金额总计_万元', ascending=False).reset_index(drop=True)
df_result3['累计核算价_万元'] = df_result3['核算价金额总计_万元'].cumsum()
df_result3['80%收入额_万元'] = df_result3['核算价金额总计_万元'].sum() * 0.8
df_result3['是否主力型号'] = df_result3['累计核算价_万元'] <= df_result3['80%收入额_万元']
df_result3

In [ ]:
revenue_median = df_result3['核算价金额总计_万元'].median()
revenue_mean = df_result3['核算价金额总计_万元'].mean()
revenue_median,revenue_mean

In [ ]:
# 四象限分析
revenue_median = df_result3['核算价金额总计_万元'].median()
revenue_mean = df_result3['核算价金额总计_万元'].mean()
def classify_revenue(row,standard):
    if row['核算价金额总计_万元'] >= standard and row['产品状态'] in ['量产']:
        return '明星产品-高收入量产产品'
    if row['核算价金额总计_万元'] < standard and row['产品状态'] in ['量产']:
        return '潜力产品-低收入量产产品'
    if row['核算价金额总计_万元'] >= standard and not row['产品状态'] in ['量产']:
        return '风险产品-高收入非量产产品'
    if row['核算价金额总计_万元'] < standard and not row['产品状态'] in ['量产']:
        return '问题产品-低收入非量产产品'
df_result3['产品分类(中位数)'] = df_result3.apply(lambda row: classify_revenue(row,revenue_median), axis=1)
df_result3['产品分类(均值)'] = df_result3.apply(lambda row: classify_revenue(row,revenue_mean), axis=1)
df_result3


In [ ]:
with pd.ExcelWriter(fr'C:\Users\zhangbon\Desktop\探索分析.xlsx') as writer:
    df_result3.to_excel(writer, sheet_name='探索', index=False)

In [ ]:
# 看一下非主力型号数据
df_result3.groupby('物料组描述',as_index=False).agg(
                                    非主力型号数 = ('是否主力型号', lambda x: (x == False).sum()),
                                    主力型号数 = ('是否主力型号', lambda x: (x == True).sum()),
                                    型号数 = ('是否主力型号', 'count'),
)

### 产品效率模型

In [ ]:
df_temp3 = df1.copy()
df_temp3

In [ ]:
df_temp3[df_temp3['物料编码']=='1002003400289']

#### 产品级指标

In [ ]:
# 从最细粒度汇总到产品级
产品级指标 = df_temp3.groupby('物料编码',as_index=False).agg(
    实际出库数量=('实际出库数量','sum'),           # 总发货量
    系统核算价_万元=('系统核算价_万元','mean'),            # 系统核算价
    核算价_万元=('核算价_万元','sum'),            # 总收入
    渠道=('渠道','nunique'),              # 覆盖渠道数
    发货月份=('发货月份','nunique'),          # 活跃月份数
    产品状态=('产品状态','first'),            # 当前状态
    标准型号=('标准型号','first'),            # 标准型号
    产品线=('产品线','first'),              # 所属产品线
    开始销售时间=('开始销售时间','first')         # 上市时间    
).reset_index()
# 派生指标
产品级指标['月均销售额_万元'] = 产品级指标['核算价_万元'] / 产品级指标['发货月份']
产品级指标['渠道覆盖率'] = 产品级指标['渠道'] / 产品级指标['渠道'].nunique()
产品级指标

#### 产品&渠道分析

In [ ]:
# 分析每个产品在各个渠道的表现
渠道级指标 = df_temp3.groupby(['物料编码', '渠道']).agg({
    '实际出库数量': 'sum',
    '核算价_万元': 'sum',
    '发货月份': 'nunique'
}).reset_index()

# 计算渠道贡献度
渠道级指标['渠道月均销售额_万元'] = 渠道级指标['核算价_万元'] / 渠道级指标['发货月份']

# 渠道依赖度分析
c = 渠道级指标.pivot_table(
    index='物料编码', 
    columns='渠道', 
    values='核算价_万元', 
    fill_value=0
)
c

#### 产品&时序分析

In [ ]:
# 月度销售趋势
月度趋势 = df_temp3.groupby(['物料编码', '发货月份']).agg({
    '实际出库数量': 'sum',
    '核算价_万元': 'sum'
}).reset_index()

# 计算月度增长率
月度趋势 = 月度趋势.sort_values(['物料编码', '发货月份'])
月度趋势['月度增长率'] = 月度趋势.groupby('物料编码')['核算价_万元'].pct_change()
月度趋势